# TinkerPop 3.7 vs 4.0: GLV Driver Benchmarks (ledger edition)

**Environment:** Server m7a.8xlarge (us-east-2), Client m7a.4xlarge (us-west-2), Gremlin Server standalone, Modern graph.

**Protocol:** 3.7 = WebSocket (multiplexed for Java/.NET/JS), 4.0 = HTTP (connection-per-request).

**Server config:** `threadPoolWorker: 8`, `gremlinPool: 16`, `maxWorkQueueSize: 65536` (identical tuning for both versions).

**Concurrency model:** "Effective concurrency" = number of requests simultaneously in-flight to the server.

> **Data source.** This notebook reads the **append-only `ledger.csv`** produced by the
> `bench` harness (one wide row per measured cell), *not* the legacy `results.csv`.
> Version is not a column in the ledger — it is recovered from `git_sha`
> (see `VER` below). Throughput/latency use the **median** of each cell's
> executions (the harness's robust primary signal), where the old report used the mean.

In [ ]:
!pip install plotly kaleido nbformat boto3 pandas -q

import boto3
# The bench harness's append-only ledger, published to S3 by the EC2 client.
# Override LEDGER_KEY if you sync it under a different key (e.g. 'go-rerun/ledger.csv').
BUCKET     = 'kirill-tp-benchmarks'
LEDGER_KEY = 'ledger.csv'
boto3.client('s3').download_file(BUCKET, LEDGER_KEY, 'ledger.csv')
print(f"Downloaded s3://{BUCKET}/{LEDGER_KEY} -> ledger.csv")

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- Version recovery --------------------------------------------------------
# The ledger does not carry a version column; each row is stamped with the
# git_sha of the branch that produced it. Map the two control SHAs to versions.
# 8fe55a2...=4.0 branch (4-glv-python-perf), 5913d57...=3.7 branch (3.7-glv-benchmarking).
# Add SHAs here if a fresh Go 4.0 re-run lands on a new commit.
VER = {
    '8fe55a250743e76dbfaa05e5cac23d86ff614d98': '4.0',
    '5913d5728388d227f7578041b9ccec4d8f0f12a3': '3.7',
}
# If a Go 4.0 re-run was committed on a newer 4.0 SHA, list it here so it maps to 4.0:
EXTRA_40_SHAS = []   # e.g. ['c7d9d3d27b6e997244797a15ae0e1d4877512a20']
for s in EXTRA_40_SHAS:
    VER[s] = '4.0'

CATEGORY = {
    'protocol-overhead': 'cat1-protocol-overhead',
    'peak-throughput':   'cat3-peak',
    'scaling-curve':     'cat4-scaling-curve',
    'pool-sensitivity':  'cat5-pool-sensitivity',
}

raw = pd.read_csv('ledger.csv')
raw = raw[raw['status'] == 'ok'].copy()
raw['version'] = raw['git_sha'].map(VER)
unmapped = sorted(raw[raw['version'].isna()]['git_sha'].unique())
if unmapped:
    print('WARNING: dropping rows with unmapped git_sha (add to VER):')
    for s in unmapped:
        print('  ', s)
raw = raw.dropna(subset=['version'])

# --- Normalize the wide ledger into the legacy report's column shape ---------
# so the Section 1-4 chart code reads almost identically to the old report.ipynb.
df = pd.DataFrame({
    'version':         raw['version'].astype(str),
    'category':        raw['test'].map(CATEGORY),
    'glv':             raw['glv'],
    'test_size':       raw['point_value'].where(raw['test'] == 'protocol-overhead'),
    'concurrency':     pd.to_numeric(raw['concurrency'], errors='coerce'),
    'pool_size':       pd.to_numeric(raw['pool'], errors='coerce'),
    # MEDIAN is the harness's robust signal: throughput rows -> req/s, latency rows -> sec.
    'avg_req_sec':     pd.to_numeric(raw['median'].where(raw['metric'] == 'throughput'), errors='coerce'),
    'avg_latency_sec': pd.to_numeric(raw['median'].where(raw['metric'] == 'latency'), errors='coerce'),
})

GLV_ORDER = ['java', 'go', 'dotnet', 'javascript', 'python']
GLV_LABELS = {'java': 'Java', 'go': 'Go', 'dotnet': '.NET', 'javascript': 'JavaScript', 'python': 'Python'}
VERSION_COLORS = {'3.7': '#636EFA', '4.0': '#EF553B'}
GLV_COLORS = {
    'java': '#636EFA', 'go': '#00CC96', 'dotnet': '#AB63FA',
    'javascript': '#FFA15A', 'python': '#EF553B',
}

# Pre-compute cat4 for reuse
cat4 = df[df['category'] == 'cat4-scaling-curve'].copy()
cat4['concurrency'] = cat4['concurrency'].astype(int)

print(f"{len(df)} data points loaded")
print(f"Versions: {sorted(df['version'].unique())}")
print(f"Categories: {sorted(df['category'].dropna().unique())}")
print(f"GLVs: {sorted(df['glv'].unique())}")

## Section 1: Latency (3.7 vs 4.0)

Sequential requests, pool=1, no concurrency. Raw per-request cost of each transport.
- **Tiny:** `g.V()` — 6 results, measures connection overhead + minimal serialization
- **Medium:** `g.V().repeat(both()).times(12)` — ~354K results, measures serialization/deserialization throughput

In [ ]:
cat1 = df[df['category'] == 'cat1-protocol-overhead'].copy()
cat1 = cat1[cat1['test_size'].isin(['tiny', 'medium'])]

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Tiny: g.V() — 6 results', 'Medium: repeat(both()).times(12) — ~354K results'],
    horizontal_spacing=0.12,
)

for col_idx, size in enumerate(['tiny', 'medium'], 1):
    size_data = cat1[cat1['test_size'] == size]
    for version in ['3.7', '4.0']:
        vdata = size_data[size_data['version'] == version]
        if vdata.empty:
            continue
        vdata = vdata.set_index('glv').reindex(GLV_ORDER).dropna(subset=['avg_latency_sec']).reset_index()
        fig.add_trace(
            go.Bar(
                x=[GLV_LABELS.get(g, g) for g in vdata['glv']],
                y=vdata['avg_latency_sec'],
                name=version,
                marker_color=VERSION_COLORS[version],
                showlegend=(col_idx == 1),
            ),
            row=1, col=col_idx,
        )

fig.update_layout(
    title='Per-Request Latency: 3.7 (WS) vs 4.0 (HTTP) — lower is better',
    barmode='group',
    height=450,
    legend=dict(title='Version'),
)
fig.update_yaxes(title_text='Latency (sec)', col=1)
fig.update_yaxes(title_text='Latency (sec)', col=2)
fig.show()

pivot = cat1.pivot_table(index=['glv', 'test_size'], columns='version', values='avg_latency_sec')
if '3.7' in pivot.columns and '4.0' in pivot.columns:
    pivot['delta_pct'] = ((pivot['4.0'] - pivot['3.7']) / pivot['3.7'] * 100).round(1)
display(pivot.round(4))

## Section 2: Peak Throughput (3.7 vs 4.0)

Maximum observed throughput per GLV across all test configurations (cat3 peak + cat4 scaling curve).
This answers: "What's the best each GLV can achieve when optimally tuned?"

- Java peak from cat3 (5M requests, optimal pool config for each version).
- All other GLVs peak from cat4 (max across concurrency sweep).

In [ ]:
# Derive peak from max observed across ALL throughput categories (cat3 + cat4 + cat5)
throughput_data = df[df['avg_req_sec'].notna() & (df['avg_req_sec'] > 0)].copy()
peak = throughput_data.groupby(['glv', 'version'])['avg_req_sec'].max().reset_index()
peak['glv'] = pd.Categorical(peak['glv'], categories=GLV_ORDER, ordered=True)
peak = peak.sort_values('glv')
peak['glv_label'] = peak['glv'].map(GLV_LABELS)

fig = px.bar(
    peak,
    x='glv_label', y='avg_req_sec', color='version',
    barmode='group',
    title='Peak Throughput per GLV — higher is better',
    labels={'avg_req_sec': 'Requests/sec', 'glv_label': 'Driver'},
    color_discrete_map=VERSION_COLORS,
)
fig.update_layout(height=450, legend_title='Version')
fig.show()

pivot = peak.pivot_table(index='glv', columns='version', values='avg_req_sec', observed=False)
if '3.7' in pivot.columns and '4.0' in pivot.columns:
    pivot['delta_pct'] = ((pivot['4.0'] - pivot['3.7']) / pivot['3.7'] * 100).round(1)
display(pivot)

## Section 3: Throughput Regression/Improvement (4.0 vs 3.7)

Two views of the same question: "Did 4.0 regress?"

- **Ratio chart (left):** `4.0 / 3.7` throughput ratio at each concurrency level. Line at 1.0 = parity. Above = improvement, below = regression.
- **Delta % bar chart (right):** Single-number summary at C=256.

> Java 3.7 is bottlenecked by client config — its ratio is artificially inflated and excluded.

In [ ]:
# --- Ratio chart: 4.0/3.7 throughput at each concurrency level ---
# Exclude Java (3.7 is bottlenecked, ratio would be meaninglessly inflated)
cat4_valid = cat4[cat4['glv'] != 'java'].copy()
shared_c = sorted(set(cat4_valid[cat4_valid['version']=='3.7']['concurrency']) &
                  set(cat4_valid[cat4_valid['version']=='4.0']['concurrency']))

ratio_data = cat4_valid[cat4_valid['concurrency'].isin(shared_c)].pivot_table(
    index=['glv', 'concurrency'], columns='version', values='avg_req_sec').reset_index()
ratio_data = ratio_data.dropna(subset=['3.7', '4.0'])
ratio_data['ratio'] = ratio_data['4.0'] / ratio_data['3.7']

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Throughput Ratio (4.0 / 3.7) vs Concurrency', 'Delta % at C=256'],
    column_widths=[0.65, 0.35],
    horizontal_spacing=0.12,
)

for glv in GLV_ORDER:
    gdata = ratio_data[ratio_data['glv'] == glv].sort_values('concurrency')
    if gdata.empty:
        continue
    fig.add_trace(go.Scatter(
        x=gdata['concurrency'], y=gdata['ratio'],
        mode='lines+markers', name=GLV_LABELS.get(glv, glv),
        line=dict(color=GLV_COLORS.get(glv)),
    ), row=1, col=1)

fig.add_hline(y=1.0, line_dash='dash', line_color='gray',
              annotation_text='parity (1.0)', annotation_position='bottom right',
              row=1, col=1)

# Right panel: delta % bar at C=256
c256_ratio = ratio_data[ratio_data['concurrency'] == 256].copy()
c256_ratio['delta_pct'] = (c256_ratio['ratio'] - 1) * 100
c256_ratio['glv'] = pd.Categorical(c256_ratio['glv'], categories=GLV_ORDER, ordered=True)
c256_ratio = c256_ratio.sort_values('glv')
c256_ratio['glv_label'] = c256_ratio['glv'].map(GLV_LABELS)

colors = ['#2ca02c' if d >= 0 else '#d62728' for d in c256_ratio['delta_pct']]
fig.add_trace(go.Bar(
    x=c256_ratio['glv_label'], y=c256_ratio['delta_pct'],
    marker_color=colors,
    showlegend=False,
    text=[f"{d:+.0f}%" for d in c256_ratio['delta_pct']],
    textposition='outside',
), row=1, col=2)

fig.add_hline(y=0, line_dash='dash', line_color='gray', row=1, col=2)

fig.update_layout(
    title='Throughput Change: 4.0 vs 3.7 — above line = improvement (Java excluded)',
    height=450,
    legend_title='Driver',
)
fig.update_xaxes(title_text='Effective Concurrency', type='log',
                 tickvals=[4, 16, 64, 128, 256, 512, 1000], row=1, col=1)
fig.update_yaxes(title_text='Ratio (4.0 / 3.7)', row=1, col=1)
fig.update_yaxes(title_text='Delta %', row=1, col=2)
fig.show()

print("Throughput ratio at C=256 (4.0 / 3.7):")
display(c256_ratio[['glv_label', 'ratio', 'delta_pct']].rename(
    columns={'glv_label': 'Driver', 'ratio': '4.0/3.7 Ratio', 'delta_pct': 'Delta %'}
).set_index('Driver').round(2))

## Section 4: Cross-GLV Scaling (3.7 vs 4.0)

All GLVs overlaid on one chart at the same concurrency levels.

- **3.7 (left):** GLVs diverge — JS dominates via WS multiplexing, Go/Python plateau early.
- **4.0 (right):** Java, Go, and .NET converge — the bottleneck is the server, not the client driver.

> Java 3.7 is bottlenecked by client WS config, not representative of true WS performance.

In [ ]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['3.7 (WebSocket)', '4.0 (HTTP)'],
    horizontal_spacing=0.10,
)

# 3.7 panel — all GLVs
cat4_37 = cat4[cat4['version'] == '3.7']
for glv in GLV_ORDER:
    vdata = cat4_37[cat4_37['glv'] == glv].sort_values('concurrency')
    if vdata.empty:
        continue
    fig.add_trace(go.Scatter(
        x=vdata['concurrency'], y=vdata['avg_req_sec'],
        mode='lines+markers', name=GLV_LABELS.get(glv, glv),
        line=dict(color=GLV_COLORS.get(glv)),
        showlegend=True,
    ), row=1, col=1)

# 4.0 panel — all GLVs
cat4_40 = cat4[cat4['version'] == '4.0']
for glv in GLV_ORDER:
    vdata = cat4_40[cat4_40['glv'] == glv].sort_values('concurrency')
    if vdata.empty:
        continue
    fig.add_trace(go.Scatter(
        x=vdata['concurrency'], y=vdata['avg_req_sec'],
        mode='lines+markers', name=GLV_LABELS.get(glv, glv),
        line=dict(color=GLV_COLORS.get(glv)),
        showlegend=False,
    ), row=1, col=2)

fig.update_layout(
    title='Cross-GLV Scaling: All Drivers at Same Concurrency',
    height=500,
    legend_title='Driver',
)
fig.update_xaxes(title_text='Effective Concurrency', type='log',
                 tickvals=[4, 16, 64, 128, 256, 512, 1000, 5000])
fig.update_yaxes(title_text='req/sec', type='log')
fig.show()

## Section 5: Pool Sensitivity (3.7 vs 4.0)

Throughput vs. connection-pool size at fixed high parallelism. This is the
**key 4.0 diagnostic** (per `BENCHMARKING.md`): because 4.0/HTTP is
connection-per-request, pool size ≈ effective concurrency, so the curve is
**steep for every GLV** — an undersized pool is the throughput limiter. Under
3.7/WS, multiplexing (Java/.NET/JS) flattens the curve; Go/Python stay steep.

> This section is new to the ledger edition — the legacy `results.csv` did not
> retain the pool-sensitivity sweep.

In [ ]:
cat5 = df[df['category'] == 'cat5-pool-sensitivity'].copy()
cat5['pool_size'] = cat5['pool_size'].astype(int)

if cat5.empty:
    print('No pool-sensitivity rows in this ledger.')
else:
    glvs_present = [g for g in GLV_ORDER if g in set(cat5['glv'])]
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=['3.7 (WebSocket)', '4.0 (HTTP)'],
        horizontal_spacing=0.10,
    )
    for col_idx, version in enumerate(['3.7', '4.0'], 1):
        vall = cat5[cat5['version'] == version]
        for glv in glvs_present:
            gdata = vall[vall['glv'] == glv].sort_values('pool_size')
            if gdata.empty:
                continue
            fig.add_trace(go.Scatter(
                x=gdata['pool_size'], y=gdata['avg_req_sec'],
                mode='lines+markers', name=GLV_LABELS.get(glv, glv),
                line=dict(color=GLV_COLORS.get(glv)),
                showlegend=(col_idx == 1),
            ), row=1, col=col_idx)

    fig.update_layout(
        title='Pool Sensitivity: Throughput vs Connection-Pool Size',
        height=450,
        legend_title='Driver',
    )
    fig.update_xaxes(title_text='Connection-pool size', type='log')
    fig.update_yaxes(title_text='req/sec', col=1)
    fig.update_yaxes(title_text='req/sec', col=2)
    fig.show()

    # Steepness summary: max/min throughput ratio across the sweep per GLV/version
    # (a flat curve ~ 1.0; a steep curve >> 1.0).
    span = (cat5.groupby(['glv', 'version'])['avg_req_sec']
                .agg(['min', 'max']))
    span['steepness (max/min)'] = (span['max'] / span['min']).round(2)
    display(span.round(1))

## Summary

| Question | Where to look | What to read off the charts |
|----------|---------------|------------------------------|
| Does 4.0 HTTP add latency? | Section 1 (Latency) | Compare the tiny/medium bars + the `delta_pct` table per GLV. |
| What's the peak per GLV? | Section 2 (Peak) | Tallest bar per driver; `delta_pct` = 4.0 vs 3.7. |
| Did 4.0 regress? | Section 3 (Regression) | Ratio above 1.0 = 4.0 wins; the C=256 delta bar is the single-number summary. |
| Which GLV to choose? | Section 4 (Cross-GLV) | Where the 4.0 curves converge = server-bound, not driver-bound. |
| Is the pool size hurting me? | Section 5 (Pool sensitivity) | Steep curve ⇒ pool size is the limiter (expected for all 4.0 GLVs). |

**Go focus (this re-run).** Read Go's line in Sections 3–5: the 4.0/3.7 ratio
across concurrency (Section 3), Go vs the converged Java/.NET cluster
(Section 4), and Go's pool-sensitivity steepness (Section 5).

**Method note.** Values are the **median** of each cell's executions (the
harness's robust signal), versions are recovered from `git_sha` via `VER`, and
only `status == 'ok'` rows are charted. A fresh Go 4.0 re-run on a new commit
just needs its SHA added to `EXTRA_40_SHAS` in the load cell.

**Known limitation:** Java 3.7 cat4 is bottlenecked by a WS client config issue
(nioPoolSize/parallelism/maxSimultaneous interaction), so Java is excluded from
the Section 3 ratio. Its true WS peak (~35K, Section 2 cat3) is validated separately.